# 🧬 BioKG Tutorial Part 3: Architecting and Training RotatE

In Part 2, we mapped high-dimensional biological properties to vectors of uniform size. 

Now we inject that state-of-the-art biological knowledge natively into our **RotatE** Knowledge Graph Embedding Model and train it using **PyTorch Lightning**! By the end of this run, the model will be able to perform true Link Prediction: guessing interacting Drugs for untreated Diseases.

---

## ⚡ Before Starting
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Run all cells

## 📦 Step 1: Initialize Core Frameworks
We use **Lightning** to handle GPU scaling and Epoch loops automatically. We use **MLflow** for enterprise experiment tracking (logging out loss curves independently in the background).

In [ ]:
!pip install lightning mlflow loguru rich -q

import sys
sys.path.append('.')

print("✅ Frameworks Loaded.")

## 🧠 Step 2: The RotatE Algorithm

Knowledge Graphs don't train like typical Neural Networks (where you pass in X and look for Y). KGE uses **Negative Sampling Optimization**.

### How RotatE Learns:
1. Pull a real relationship from PrimeKG: `[Aspirin] -> [Indication_For] -> [Headache]`
2. Translate them to complex math: `Entity_Vector(Aspirin) ∘ Rotation_Vector(Indication) ≈ Entity_Vector(Headache)`
3. Is the math equation close to perfect? If yes, score is high.
4. **Negative Sample**: We then corrupt either the Head or Tail with a random node: `[Aspirin] -> [Indication_For] -> [Broken_Toe]`.
5. We train the network to drive the score for the False relationship as low as possible!

### 16-Mixed Precision
Because we are processing 8.1M triples along with our huge BioBridge tensors, we will instruct PyTorch Lightning to use **BF-16 Precision** (Brain floating-point 16). This halves the required GPU caching, letting us pass Massive **2,048 element** batch sizes directly into the GPU streaming cores!

In [ ]:
from training.trainer import train

# Custom Configuration Overrides for the Tutorial
tutorial_config = {
    "batch_size": 2048,           # Massive payload utilizing our 16-bit caching
    "num_workers": 2,             # Streamlines data loading over CPU threads
    "train_fraction": 0.25,       # Use 25% of PrimeKG just for rapid speed in this demo!
    "embed_dim": 384,             # Complex RotatE math doubles this to align with our 768 Projector!
    "gamma": 12.0,                # Margin boundary separating real vs fake pairs
    "n_negative_samples": 64,     # Generate 64 negative falsified graphs per positive
    "use_biobridge": True,        # ACTIVATE the Multimodal pre-computed initialization!
    "lr": 5e-4,
    "max_epochs": 2,              # Demo 2 epochs
    "precision": "16-mixed",      # Vital for fitting 2048 arrays
    "gradient_clip_val": 1.0,     # Prevent exploding gradient math
    "experiment_name": "BioKG-PrimeKG-LinkPrediction",
    "run_name": "tutorial_rotate_demo",
}

print("🚀 Launching the BioKG PyTorch Lightning Training Engine...")
best_model_path = train(config=tutorial_config)
print(f"\n📁 Best Model checkpoint automatically stored locally at: {best_model_path}")

## 📈 Step 3: Track Metrics live using MLflow server tunneling
Even inside Colab, we can expose port 5000 via a safe Localtunnel to utilize the official standard Dashboard UI of **MLflow**! This is exactly how industry metrics are documented.

In [ ]:
import subprocess

# Boot MLflow background server linked directly to our outputs dir
!npm install -g localtunnel -q
!nohup mlflow ui --port 5000 > mlflow.log 2>&1 &

print("✅ MLflow Running On Port 5000 internally.")
print("🌐 Tunneling port out now! Click the resulting link below:\n")

# Note: The link will ask for an endpoint IP password (usually printed below command)
!npx localtunnel --port 5000

### 🎉 Summary of Part 3
You have successfully trained a state-of-the-art Deep Learning KGE model integrated with bleeding-edge NLP and Multi-omic embeddings on a Google Colab instance!

Because the Checkpoint (`.ckpt`) is saved over 200 Megabytes on your side, you can now instantiate the **FastAPI Inference Server (`app.py`)** found in the main repository to provide real-time Graph searches via UI to your clients/recruiters.